# M3 first submission: copy the validated M2 package only
Before execution, select **Accelerator: None**, **Internet: Off**. Kaggle auto-extracts uploaded `.zip` files in the observed dataset workflow. Upload a byte-identical copy of the M2 package named `submission.zip.bin`, together with the unchanged committed `manifest.json`, as a private input. Do not rename or edit the original M2 evidence. Set MANIFEST_PATH and PACKAGE_PATH to the actual mounted files and set AUTHORIZATION below. The input filename does not replace any identity check.

This notebook performs no installation, compilation, extraction, model serving, evaluation or submission API call. The only output write is `/kaggle/working/submission.zip`, copied byte-for-byte. The approval phrase authorizes this preparation; actual competition submission is a separate operator action. Existing output is never overwritten. Stored official sources do not fully document the live UI selection sequence; review it before any submission. A scored submission would execute the agent on competition infrastructure.


In [1]:
import hashlib
import json
from pathlib import Path

AUTHORIZATION_REQUIRED = "AUTHORIZE FORGEAI GEMMA4 M3 FIRST SUBMISSION OF M2 PACKAGE ca6a05b4"
MANIFEST_SHA256 = "91660911e8122cf4e6080add5d52972072f69eb13b8e7f048d1dfac108afac37"
ZIP_SHA256 = "78051300426d369e83a788f57bc6284c56c85d07cc32d6de87fbdc49c9da7e2d"
RECEIPT_SHA256 = "ca6a05b46844c86369c7aaef3d989b542c5f3c5f63fcc91c7fce5997742893d9"
ZIP_SIZE = 1566


def prepare_submission(manifest_path, package_path, output, authorization):
    if authorization != AUTHORIZATION_REQUIRED:
        raise ValueError("Exact maintainer authorization required")
    if manifest_path.is_symlink() or package_path.is_symlink():
        raise ValueError("Evidence files must not be symlinks")
    raw_manifest = manifest_path.read_bytes()
    if hashlib.sha256(raw_manifest).hexdigest() != MANIFEST_SHA256:
        raise ValueError("Committed M2 manifest SHA256 mismatch")
    manifest = json.loads(raw_manifest)
    if (manifest.get("state") != "PORTABLE_COMPILE_AND_PACKAGE_PASS"
            or manifest.get("local_receipt_sha256") != RECEIPT_SHA256
            or manifest.get("zip_sha256") != ZIP_SHA256
            or manifest.get("zip_size") != ZIP_SIZE):
        raise ValueError("M2 package identity mismatch")
    payload = package_path.read_bytes()
    if len(payload) != ZIP_SIZE or hashlib.sha256(payload).hexdigest() != ZIP_SHA256:
        raise ValueError("M2 submission.zip SHA256/size mismatch")
    # Exclusive creation: preserve prior output; no repacking or extraction.
    with output.open("xb") as stream:
        stream.write(payload)
    if hashlib.sha256(output.read_bytes()).hexdigest() != ZIP_SHA256:
        raise ValueError("Copied output SHA256 mismatch; do not submit")
    report = {"state": "M2_PACKAGE_COPIED_UNCHANGED", "output": str(output),
              "zip_sha256": ZIP_SHA256, "zip_size": ZIP_SIZE,
              "manifest_sha256": MANIFEST_SHA256, "receipt_sha256": RECEIPT_SHA256}
    print(json.dumps(report, indent=2, sort_keys=True))
    return report


In [4]:
AUTHORIZATION = "AUTHORIZE FORGEAI GEMMA4 M3 FIRST SUBMISSION OF M2 PACKAGE ca6a05b4"
MANIFEST_PATH = Path("/kaggle/input/datasets/nishanthrajandev/forgeai-gemma4-m2-package-ca6a05b4/manifest.json")
PACKAGE_PATH = Path("/kaggle/input/datasets/nishanthrajandev/forgeai-gemma4-m2-package-ca6a05b4/submission.zip.bin")
OUTPUT = Path("/kaggle/working/submission.zip")
report = prepare_submission(MANIFEST_PATH, PACKAGE_PATH, OUTPUT, AUTHORIZATION)


{
  "manifest_sha256": "91660911e8122cf4e6080add5d52972072f69eb13b8e7f048d1dfac108afac37",
  "output": "/kaggle/working/submission.zip",
  "receipt_sha256": "ca6a05b46844c86369c7aaef3d989b542c5f3c5f63fcc91c7fce5997742893d9",
  "state": "M2_PACKAGE_COPIED_UNCHANGED",
  "zip_sha256": "78051300426d369e83a788f57bc6284c56c85d07cc32d6de87fbdc49c9da7e2d",
  "zip_size": 1566
}


Expected: `M2_PACKAGE_COPIED_UNCHANGED` and ZIP SHA256 `78051300426d369e83a788f57bc6284c56c85d07cc32d6de87fbdc49c9da7e2d`. Save the notebook output and hash. Do not execute the official starter cells. After separate maintainer approval, use the competition interface to select this saved notebook output archive if supported; stop if the interface requirements differ. Do not select compiler logs or submission.parquet. Preserve the submitted notebook version, submission ID, timestamps, public score/status and all available scoring logs.
